In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import sys
sys.path.append('/content/drive/MyDrive/repositories/aq_edge')

In [4]:
%cd /content/drive/MyDrive/repositories/aq_edge/colab
%run mlflow_colab.py

/content/drive/MyDrive/repositories/aq_edge/colab
MLflow server ready: http://127.0.0.1:5000


In [19]:
%cd /content/drive/MyDrive/repositories/aq_edge

import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

# Ausführen des Skripts mit expliziter Übergabe der Parameter für die Station USAM und 2 Trials
!python hyperparameter_optuna.py --data.station 'USAM' --optuna.n_trials 2

/content/drive/MyDrive/repositories/aq_edge/colab
[2026-10-21 08:57:39] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261021_085739.log
[2026-10-21 08:57:39] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261021_085739.log
[2026-10-21 08:57:41] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_085741.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_085741.log
[2026-10-21 08:57:41] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-21 08:57:41] INFO: Device: cuda
INFO:__main__:Device: cuda
[2026-10-21 08:57:41] INFO: Station: UTEC
INFO:__main__:Station: UTEC
[2026-10-21 08:57:41] INFO: Features: ['TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['TEM', 'HUM', 'VOC']
[2026-10-21 08:57:41] INFO: Target: CO2
INFO:__main__:Target: CO

In [6]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

MLflow database and artifacts backed up to Drive.


In [17]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil

# Pfade definieren
base_repo_dir = Path("/content/drive/MyDrive/repositories/aq_edge")
target_base_dir = base_repo_dir / "optuna_results"
possible_dirs = [
    target_base_dir,
    base_repo_dir / "colab" / "optuna_results"
]

# Überprüfen, ob Daten im Kernel-Speicher vorhanden sind
has_kernel_data = 'best_params' in locals() and 'df_trials' in locals()

if has_kernel_data:
    print("Aktive Trainingsdaten im Kernel-Speicher gefunden! Nutze Arbeitsspeicher für den Export...")

    # Station bestimmen und Zielordner erstellen
    station_name = best_params.get("station", "UTEC").upper()
    station_dir = target_base_dir / station_name
    station_dir.mkdir(parents=True, exist_ok=True)

    # Zeitstempel generieren
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Zielpfade definieren
    target_json_path = station_dir / f"best_params_{station_name}_{timestamp_str}.json"
    target_csv_path = station_dir / f"trials_{station_name}_{timestamp_str}.csv"

    # 1. Best-Parameter aus dem Speicher exportieren
    with open(target_json_path, 'w', encoding='utf-8') as f:
        json.dump(best_params, f, indent=2, ensure_ascii=False)

    # 2. Trials-DataFrame aus dem Speicher exportieren
    df_trials.to_csv(target_csv_path, index=False)

    print(f"Aktuelle Ergebnisse erfolgreich exportiert nach: {station_dir.resolve()}")

    # 3. Alle älteren Dateien der Station aus den Quellordnern einsammeln und verschieben
    moved_files_count = 0
    for source_dir in possible_dirs:
        if source_dir.exists():
            for file_path in source_dir.iterdir():
                if file_path.is_file():
                    # Suchen nach Dateien, die den Stationsnamen im Dateinamen tragen
                    if station_name in file_path.name.upper() and file_path.suffix in ['.json', '.csv']:
                        # Verhindern, dass gerade erstellte Dateien verschoben werden
                        if file_path != target_json_path and file_path != target_csv_path:
                            dest_path = station_dir / file_path.name
                            shutil.move(str(file_path), str(dest_path))
                            moved_files_count += 1
                            print(f"Archivierte alte Datei verschoben: {file_path.name} -> {station_name}/{file_path.name}")

    print(f"\nErfolgreich {moved_files_count} ältere Datei(en) in den Stations-Ordner {station_dir.resolve()} verschoben.")

else:
    print("Fehler: Keine aktiven Trainingsdaten im Kernel-Speicher gefunden und keine Export-Dateien auf dem Laufwerk vorhanden.")

Aktive Trainingsdaten im Kernel-Speicher gefunden! Nutze Arbeitsspeicher für den Export...
Aktuelle Ergebnisse erfolgreich exportiert nach: /content/drive/MyDrive/repositories/aq_edge/optuna_results/UTEC

Erfolgreich 0 ältere Datei(en) in den Stations-Ordner /content/drive/MyDrive/repositories/aq_edge/optuna_results/UTEC verschoben.
